# Exploring whether 1 / L^2 = 1 / L_0^2 + K_DDD * DDD for GaAs across the literature




In [ ]:
# https://www.sciencedirect.com/science/article/pii/S2666386426002985
# In general, the short-circuit current (Isc) is degraded by the
# radiation damage to the diffusion length in the neutral region.
# Previous studies have investigated this,2,3 and this knowledge is
# reflected in the design of current 3J space solar cells, which show
# little Isc degradation caused by radiation

import numpy as np
import scipy.constants as const

In [ ]:
# TODO-TD: move to library

# def Lindhard_partition(energy):

# https://iopscience.iop.org/article/10.1143/JJAP.50.072301/pdf

# def NIEL(energy, T_d, T_max):

#     # TODO-TD: How to integrate this?
#     # Differntial cross section?
#     # Or lookup table?
#     summ = 0
#     for T in range(T_d, T_max):
#         summ += T * Lindhard_partition(energy) * 


def DDD_niel(fluence, niel):
    return fluence * niel

def diffusion_length(diffusion_const, carrier_lifetime):
    return np.sqrt(diffusion_const, carrier_lifetime)

def diffusion_length_fluence(L_0, K_L, fluence):
    """
    1/L^2 = 1/L_0^2 + K_L * phi

    """
    den = 1 + L_0 * L_0 * K_L * fluence
    return L_0 * np.sqrt(1 / den)

# TODO-TD: one giant p-n junction class eventually?

class W_SCR:
    def __init__(
        self,
        built_in_voltage,
        semiconductor_permittivity,
        base_doping_density,
    ):
        self.V_bi = built_in_voltage
        self.eps = semiconductor_permittivity
        self.N_A = base_doping_density

    def __call__(self, voltage):
        return np.sqrt(2 * self.eps * (self.V_bi - voltage) / (const.elementary_charge * self.N_A))
    

# TODO-TD: class that holds onto material constants, then evaled at voltage?
def multi_diode_current_model(
    voltage,
    temperature,
    absorbtion_coeff,
    photon_flux,
    emitter_minority_carrier_length,
    base_minority_carrier_length,
    scr_location,
    reverse_saturation_current_01,
    reverse_saturation_current_02,
    ideality_factor,
    w_scr: W_SCR,
):
    """
    Salzberger2018 equations
    """
    flux_indpnt_current = diode_current(
        voltage, 
        reverse_saturation_current_01, 
        temperature,
    ) + diode_current(
        voltage, 
        reverse_saturation_current_02, 
        temperature,
        ideality_factor,
    )
    i_phi_w = shape_charge_region_current(
        voltage,
        absorbtion_coeff,
        base_minority_carrier_length,
        scr_location,
        w_scr,
    )
    i_emitter = emitter_current(
        absorbtion_coeff,
        emitter_minority_carrier_length,
        scr_location,
    )
    return const.elementary_charge * photon_flux * (i_emitter + i_phi_w) - flux_indpnt_current


def emitter_current(
    absorbtion_coeff,
    emitter_minority_carrier_length,
    scr_location,
):
    a =  (np.exp(-scr_location / emitter_minority_carrier_length) - np.exp(-absorbtion_coeff * scr_location))
    b = absorbtion_coeff  * emitter_minority_carrier_length / (absorbtion_coeff  * emitter_minority_carrier_length - 1)
    return a * b

def shape_charge_region_current(
    voltage,
    absorbtion_coeff,
    base_minority_carrier_length,
    scr_location,
    w_scr: W_SCR,
):
    
    a = np.exp(-absorbtion_coeff * scr_location)
    b_den = (1 + absorbtion_coeff * base_minority_carrier_length)
    b = 1 - (np.exp(-absorbtion_coeff * w_scr(voltage))) / b_den
    return a * b


def diode_current(
    voltage,
    reverse_saturation_current,
    temperature,
    ideality_factor=1,
):
    """
    Standard diode equation
    """
    diode = np.exp(const.e * voltage / (ideality_factor * const.Boltzmann * temperature))
    return reverse_saturation_current * (diode - 1)

